In [25]:
#Import necessary libraries
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [26]:
# Cardiovascular paths
PATH_CARDIO_TRAIN = "../../cardiovascularData/cardio_train.csv"
PATH_FRAMINGHAM   = "../../cardiovascularData/framingham.csv"
PATH_CLEVELAND    = "../../cardiovascularData/processed.cleveland.data"

paths = {
    'cardio_train':   PATH_CARDIO_TRAIN,
    'framingham':     PATH_FRAMINGHAM,
    'cleveland':      PATH_CLEVELAND
}

for name, path in paths.items():
    if os.path.exists(path):
        print(f'  OK       {name}')
    else:
        print(f'  MISSING  {name}  --> {path}')

  OK       cardio_train
  OK       framingham
  OK       cleveland


In [27]:
# cardio training data

cardio = pd.read_csv(PATH_CARDIO_TRAIN, sep=';')

print('Shape:', cardio.shape)
print('\nColumn names:')
print(cardio.columns.tolist())
print('\nFirst 5 rows (RAW):')
cardio.head()

print('Missing values per column:')
print(cardio.isnull().sum())

print('\nData types:')
print(cardio.dtypes)

print('\nBasic stats:')
cardio.describe()


Shape: (70000, 13)

Column names:
['id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']

First 5 rows (RAW):
Missing values per column:
id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64

Data types:
id               int64
age              int64
gender           int64
height           int64
weight         float64
ap_hi            int64
ap_lo            int64
cholesterol      int64
gluc             int64
smoke            int64
alco             int64
active           int64
cardio           int64
dtype: object

Basic stats:


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
count,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,49972.419900,19468.865814,1.349571,164.359229,74.205690,128.817286,96.630414,1.366871,1.226457,0.088129,0.053771,0.803729,0.499700
std,28851.302323,2467.251667,0.476838,8.210126,14.395757,154.011419,188.472530,0.680250,0.572270,0.283484,0.225568,0.397179,0.500003
min,0.000000,10798.000000,1.000000,55.000000,10.000000,-150.000000,-70.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,25006.750000,17664.000000,1.000000,159.000000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
50%,50001.500000,19703.000000,1.000000,165.000000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,74889.250000,21327.000000,2.000000,170.000000,82.000000,140.000000,90.000000,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000
max,99999.000000,23713.000000,2.000000,250.000000,200.000000,16020.000000,11000.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000


In [28]:
#cleaning cardio training

cardio['age_years'] = (cardio['age'] / 365.25)
cardio.drop(columns=['age'], inplace=True)

# make sure the values are possible
# Blood pressure: keep systolic 80-250, diastolic 40-150
cardio = cardio[(cardio['ap_hi'] >= 80) & (cardio['ap_hi'] <= 250)]
cardio = cardio[(cardio['ap_lo'] >= 40) & (cardio['ap_lo'] <= 150)]
cardio = cardio[(cardio['height'] >= 100) & (cardio['height'] <= 220)]
cardio = cardio[(cardio['weight'] >= 30) & (cardio['weight'] <= 200)]

cardio.drop_duplicates(inplace=True)

cardio.rename(columns={'cardio': 'cvd_label'}, inplace=True)

#id provides no analytical value
if 'id' in cardio.columns:
    cardio.drop(columns=['id'], inplace=True)

print('Shape after cleaning:', cardio.shape)
print('\nCleaned data:')
cardio.head()

Shape after cleaning: (68705, 12)

Cleaned data:


,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cvd_label,age_years
0,2,168,62.0,110,80,1,1,0,0,1,0,50.357290
1,1,156,85.0,140,90,3,1,0,0,1,1,55.381246
2,1,165,64.0,130,70,3,1,0,0,0,1,51.627652
3,2,169,82.0,150,100,1,1,0,0,1,1,48.249144
4,1,156,56.0,100,60,1,1,0,0,0,0,47.841205


In [29]:
framingham = pd.read_csv(PATH_FRAMINGHAM)

print('Shape:', framingham.shape)
print('\nColumns:', framingham.columns.tolist())
print('\nMissing values:')
print(framingham.isnull().sum())
framingham.head()

Shape: (4240, 16)

Columns: ['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'TenYearCHD']

Missing values:
male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [30]:
#cleaning framingham

framingham.dropna(thresh=int(len(framingham.columns) * 0.5), inplace=True)

numeric_cols = framingham.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if framingham[col].isnull().sum() > 0:
        framingham[col].fillna(framingham[col].median(), inplace=True)

framingham.drop_duplicates(inplace=True)

print('\nShape after cleaning:', framingham.shape)
print('Missing values after cleaning:')
framingham.head()


Shape after cleaning: (4240, 16)
Missing values after cleaning:


C:\Users\mayaw\AppData\Local\Temp\ipykernel_15352\1144755537.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  framingham[col].fillna(framingham[col].median(), inplace=True)


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## Cleveland Heart Disease

In [31]:
# Cleveland has no header row, and uses '?' for missing values
cleveland_cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                  'restecg', 'thalach', 'exang', 'oldpeak',
                  'slope', 'ca', 'thal', 'target']

cleveland = pd.read_csv(PATH_CLEVELAND, names=cleveland_cols, na_values='?')

print('Shape:', cleveland.shape)
print('\nMissing values:')
print(cleveland.isnull().sum())
cleveland.head(10)

Shape: (303, 14)

Missing values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
5,56.0,1.0,2.0,120.0,236.0,0.0,0.0,178.0,0.0,0.8,1.0,0.0,3.0,0
6,62.0,0.0,4.0,140.0,268.0,0.0,2.0,160.0,0.0,3.6,3.0,2.0,3.0,3
7,57.0,0.0,4.0,120.0,354.0,0.0,0.0,163.0,1.0,0.6,1.0,0.0,3.0,0
8,63.0,1.0,4.0,130.0,254.0,0.0,2.0,147.0,0.0,1.4,2.0,1.0,7.0,2
9,53.0,1.0,4.0,140.0,203.0,1.0,2.0,155.0,1.0,3.1,3.0,0.0,7.0,1


In [32]:
cleveland.fillna(cleveland.median(numeric_only=True), inplace=True)

cleveland['target'] = (cleveland['target'] > 0).astype(int)

cleveland.drop_duplicates(inplace=True)

print('Shape after cleaning:', cleveland.shape)
print(cleveland['target'].value_counts())
cleveland.head(10)

Shape after cleaning: (303, 14)
target
0    164
1    139
Name: count, dtype: int64


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
5,56.0,1.0,2.0,120.0,236.0,0.0,0.0,178.0,0.0,0.8,1.0,0.0,3.0,0
6,62.0,0.0,4.0,140.0,268.0,0.0,2.0,160.0,0.0,3.6,3.0,2.0,3.0,1
7,57.0,0.0,4.0,120.0,354.0,0.0,0.0,163.0,1.0,0.6,1.0,0.0,3.0,0
8,63.0,1.0,4.0,130.0,254.0,0.0,2.0,147.0,0.0,1.4,2.0,1.0,7.0,1
9,53.0,1.0,4.0,140.0,203.0,1.0,2.0,155.0,1.0,3.1,3.0,0.0,7.0,1


In [33]:
output_dir = "../final_cleaned_data_files"
os.makedirs(output_dir, exist_ok=True)

cardio.to_csv(os.path.join(output_dir, "cardio_train_cleaned.csv"), index=False)
framingham.to_csv(os.path.join(output_dir, "framingham_cleaned.csv"), index=False)
cleveland.to_csv(os.path.join(output_dir, "cleveland_cleaned.csv"), index=False)

print("Saved:")
print("  cardio_train_cleaned.csv")
print("  framingham_cleaned.csv")
print("  cleveland_cleaned.csv")

Saved:
  cardio_train_cleaned.csv
  framingham_cleaned.csv
  cleveland_cleaned.csv
